<a href="https://colab.research.google.com/github/NicDecouttere/HEC_Course_GenAI_and_Corporate_Finance/blob/main/Colabexample1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install llama-index-llms-anthropic -q
!pip install llama-index -q

In [17]:
from llama_index.llms.anthropic import Anthropic
from llama_index.llms.openai import OpenAI
from llama_index.core.tools import FunctionTool

import nest_asyncio

nest_asyncio.apply()

In [18]:
from google.colab import userdata

CLAUDE_API_KEY = userdata.get('ANTHROPIC_API_KEY')
OAI_API_KEY = userdata.get('OPENAI_API_KEY')
FMP_API_KEY = userdata.get('FINANCIAL_MODELING_PREP_API_KEY')

In [19]:
openai_llm = OpenAI(model="gpt-4o-mini", api_key=OAI_API_KEY)

In [20]:
anthropic_llm = Anthropic(model="claude-3-5-sonnet-20240620", api_key=CLAUDE_API_KEY)

In [21]:
import os
import requests

# Define the function that will retrieve the financial data

def get_stock_price(symbol):
    """
    Fetch the current stock price for the given symbol, the current stock trading volume, the average 50-day, 200-day moving average price,
    Earnings per share (EPS), price/earnings (P/E) ratio, and the next earnings Announcement.
    """
    url = f"https://financialmodelingprep.com/api/v3/quote-order/{symbol}?apikey={FMP_API_KEY}"
    response = requests.get(url)
    data = response.json()
    try:
        price = data[0]['price']
        volume = data[0]['volume']
        avgVolume = data[0]['avgVolume']
        priceAvg50 = data[0]['priceAvg50']
        priceAvg200 = data[0]['priceAvg200']
        eps = data[0]['eps']
        pe = data[0]['pe']
        earningsAnnouncement = data[0]['earningsAnnouncement']
        return {"symbol": symbol.upper(), "price": price, "volume": volume, "avgVolume": avgVolume, "priceAvg50": priceAvg50, "priceAvg200": priceAvg200, "EPS": eps, "PE": pe, "earningsAnnouncement": earningsAnnouncement}
    except (IndexError, KeyError):
        return {"error": f"Invalid symbol or data not available for {symbol}"}


## DATA FORMAT OF THE STOCK PRICE FULL DATA ENDPOINT - FREE ENDPOINT (v3)
# [
# 	{
# 		"symbol": "AAPL",
# 		"name": "Apple Inc.",
# 		"price": 145.855,
# 		"changesPercentage": 0.3751,
# 		"change": 0.545,
# 		"dayLow": 143.9,
# 		"dayHigh": 146.71,
# 		"yearHigh": 179.61,
# 		"yearLow": 124.17,
# 		"marketCap": 2307703191828,
# 		"priceAvg50": 140.8724,
# 		"priceAvg200": 147.18594,
# 		"exchange": "NASDAQ",
# 		"volume": 42609394,
# 		"avgVolume": 73638864,
# 		"open": 144.38,
# 		"previousClose": 145.31,
# 		"eps": 5.89,
# 		"pe": 24.76,
# 		"earningsAnnouncement": "2023-04-26T10:59:00.000+0000",
# 		"sharesOutstanding": 15821899776,
# 		"timestamp": 1677790784
# 	}
# ]



In [22]:
def get_company_financials(symbol):
  """
  Fetch basic financial information for the given company symbol such as the␣
  industry, the sector, the name of the company, and the market capitalization.
  """
  url = f"https://financialmodelingprep.com/api/v3/profile/{symbol}?apikey={FMP_API_KEY}"
  response = requests.get(url)
  data = response.json()
  try:
    results = data[0]
    financials = {
    "symbol": results["symbol"],
    "companyName": results["companyName"],
    "marketCap": results["mktCap"],
    "industry": results["industry"],
    "sector": results["sector"],
    "website": results["website"],
    "beta":results["beta"],
    "price":results["price"],
    }
    return financials
  except (IndexError, KeyError):
    return {"error": f"Invalid symbol or data not available for {symbol}"}

## DATA FORMAT OF THE STOCK PROFILE FULL DATA ENDPOINT - FREE ENDPOINT (v3)
# [
# 	{
# 		"symbol": "AAPL",
# 		"price": 178.72,
# 		"beta": 1.286802,
# 		"volAvg": 58405568,
# 		"mktCap": 2794144143933,
# 		"lastDiv": 0.96,
# 		"range": "124.17-198.23",
# 		"changes": -0.13,
# 		"companyName": "Apple Inc.",
# 		"currency": "USD",
# 		"cik": "0000320193",
# 		"isin": "US0378331005",
# 		"cusip": "037833100",
# 		"exchange": "NASDAQ Global Select",
# 		"exchangeShortName": "NASDAQ",
# 		"industry": "Consumer Electronics",
# 		"website": "https://www.apple.com",
# 		"description": "Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. It also sells various related services. In addition, the company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; AirPods Max, an over-ear wireless headphone; and wearables, home, and accessories comprising AirPods, Apple TV, Apple Watch, Beats products, HomePod, and iPod touch. Further, it provides AppleCare support services; cloud services store services; and operates various platforms, including the App Store that allow customers to discover and download applications and digital content, such as books, music, video, games, and podcasts. Additionally, the company offers various services, such as Apple Arcade, a game subscription service; Apple Music, which offers users a curated listening experience with on-demand radio stations; Apple News+, a subscription news and magazine service; Apple TV+, which offers exclusive original content; Apple Card, a co-branded credit card; and Apple Pay, a cashless payment service, as well as licenses its intellectual property. The company serves consumers, and small and mid-sized businesses; and the education, enterprise, and government markets. It distributes third-party applications for its products through the App Store. The company also sells its products through its retail and online stores, and direct sales force; and third-party cellular network carriers, wholesalers, retailers, and resellers. Apple Inc. was incorporated in 1977 and is headquartered in Cupertino, California.",
# 		"ceo": "Mr. Timothy D. Cook",
# 		"sector": "Technology",
# 		"country": "US",
# 		"fullTimeEmployees": "164000",
# 		"phone": "408 996 1010",
# 		"address": "One Apple Park Way",
# 		"city": "Cupertino",
# 		"state": "CA",
# 		"zip": "95014",
# 		"dcfDiff": 4.15176,
# 		"dcf": 150.082,
# 		"image": "https://financialmodelingprep.com/image-stock/AAPL.png",
# 		"ipoDate": "1980-12-12",
# 		"defaultImage": false,
# 		"isEtf": false,
# 		"isActivelyTrading": true,
# 		"isAdr": false,
# 		"isFund": false
# 	}
# ]



In [23]:
def get_key_metrics_ttm(symbol):
  """
  Get key financial metrics for a company, including revenue, net income, and price-to-earnings ratio (P/E ratio). Assess a company's financial performance and compare it to its competitors.
  """
  url = f"https://financialmodelingprep.com/api/v3/key-metrics-ttm/{symbol}?apikey={FMP_API_KEY}"
  response = requests.get(url)
  data = response.json()
  try:
    results = data[0]
    key_metrics = {
    "revenuePerShareTTM": results["revenuePerShareTTM"],
    "netIncomePerShareTTM": results["netIncomePerShareTTM"],
    "operatingCashFlowPerShareTTM": results["operatingCashFlowPerShareTTM"],
    "freeCashFlowPerShareTTM": results["freeCashFlowPerShareTTM"],
    "cashPerShareTTM": results["cashPerShareTTM"],
    "bookValuePerShareTTM": results["bookValuePerShareTTM"],
    "tangibleBookValuePerShareTTM": results["tangibleBookValuePerShareTTM"],
    "marketCapTTM": results["marketCapTTM"],
    "enterpriseValueTTM": results["enterpriseValueTTM"],
    "peRatioTTM": results["peRatioTTM"],
    "priceToSalesRatioTTM": results["priceToSalesRatioTTM"],
    "priceToFCFRatioTTM": results["pfcfRatioTTM"],
    "priceToBookRatioTTM": results["ptbRatioTTM"],
    "evToSalesTTM": results["evToSalesTTM"],
    "enterpriseValueOverEBITDATTM": results["enterpriseValueOverEBITDATTM"],
    "evToOperatingCashFlowTTM": results["evToOperatingCashFlowTTM"],
    "evToFreeCashFlowTTM": results["evToFreeCashFlowTTM"],
    "earningsYieldTTM": results["earningsYieldTTM"],
    "freeCashFlowYieldTTM": results["freeCashFlowYieldTTM"],
    "debtToEquityTTM": results["debtToEquityTTM"],
    "debtToAssetsTTM": results["debtToAssetsTTM"],
    "netDebtToEBITDATTM": results["netDebtToEBITDATTM"],
    "currentRatioTTM": results["currentRatioTTM"],
    "interestCoverageTTM": results["interestCoverageTTM"],
    "roicTTM": results["roicTTM"],
    }
    return key_metrics
  except (IndexError, KeyError):
    return {"error": f"Invalid symbol or data not available for {symbol}"}


## Format follows:
# [
# 	{
# 		"revenuePerShareTTM": 24.458048210384074,
# 		"netIncomePerShareTTM": 6.036586197112504,
# 		"operatingCashFlowPerShareTTM": 7.203132909243405,
# 		"freeCashFlowPerShareTTM": 6.433270686869992,
# 		"cashPerShareTTM": 3.980350134740222,
# 		"bookValuePerShareTTM": 3.8396918155842026,
# 		"tangibleBookValuePerShareTTM": 3.8396918155842026,
# 		"shareholdersEquityPerShareTTM": 3.8396918155842026,
# 		"interestDebtPerShareTTM": 7.200966974981038,
# 		"marketCapTTM": 2767892759466,
# 		"enterpriseValueTTM": 2848764759466,
# 		"peRatioTTM": 29.32798343618193,
# 		"priceToSalesRatioTTM": 7.209311935848182,
# 		"pocfratioTTM": 24.578319216186145,
# 		"pfcfRatioTTM": 27.40840662130769,
# 		"pbRatioTTM": 46.10810150998109,
# 		"ptbRatioTTM": 46.10810150998109,
# 		"evToSalesTTM": 7.4199528549668825,
# 		"enterpriseValueOverEBITDATTM": 23.562009507183326,
# 		"evToOperatingCashFlowTTM": 25.194254629492715,
# 		"evToFreeCashFlowTTM": 28.20922256791468,
# 		"earningsYieldTTM": 0.03409712782251166,
# 		"freeCashFlowYieldTTM": 0.0364851563177914,
# 		"debtToEquityTTM": 1.8130537213392175,
# 		"debtToAssetsTTM": 0.32617195661387666,
# 		"netDebtToEBITDATTM": 0.6688887969893719,
# 		"currentRatioTTM": 0.9815625425125837,
# 		"interestCoverageTTM": 29.863225119744545,
# 		"incomeQualityTTM": 1.1932460953989026,
# 		"dividendYieldTTM": 0.005309507577062701,
# 		"dividendYieldPercentageTTM": 0.5309507577062702,
# 		"payoutRatioTTM": 0.15797804981004643,
# 		"salesGeneralAndAdministrativeToRevenueTTM": 0,
# 		"researchAndDevelopementToRevenueTTM": 0.07649511763771283,
# 		"intangiblesToTotalAssetsTTM": 0,
# 		"capexToOperatingCashFlowTTM": -0.10687880288665629,
# 		"capexToRevenueTTM": -0.03147684622056453,
# 		"capexToDepreciationTTM": -1.030176455545137,
# 		"stockBasedCompensationToRevenueTTM": 0.027312057051620986,
# 		"grahamNumberTTM": 22.83679462709757,
# 		"roicTTM": 0.5630471938175102,
# 		"returnOnTangibleAssetsTTM": 0.2828335890257225,
# 		"grahamNetNetTTM": -11.416830608779144,
# 		"workingCapitalTTM": -2304000000,
# 		"tangibleAssetValueTTM": 60274000000,
# 		"netCurrentAssetValueTTM": -152105000000,
# 		"investedCapitalTTM": 1.8130537213392175,
# 		"averageReceivablesTTM": 37542500000,
# 		"averagePayablesTTM": 44822000000,
# 		"averageInventoryTTM": 7416500000,
# 		"daysSalesOutstandingTTM": 37.25360935371536,
# 		"daysPayablesOutstandingTTM": 78.5066807297448,
# 		"daysOfInventoryOnHandTTM": 12.357922226265103,
# 		"receivablesTurnoverTTM": 9.79770836523248,
# 		"payablesTurnoverTTM": 4.649285851945438,
# 		"inventoryTurnoverTTM": 29.535709427288804,
# 		"roeTTM": 1.649211812157629,
# 		"capexPerShareTTM": -0.769862222373413,
# 		"dividendPerShareTTM": 0.94,
# 		"debtToMarketCapTTM": 0.03948129840878771
# 	}
# ]



In [24]:
def get_income_statement(symbol):
  """
  Fetch last income statement for the given company symbol such as revenue,␣
  gross profit, net income, EBITDA, EPS.
  """
  url = f"https://financialmodelingprep.com/api/v3/income-statement/{symbol}?period=annual&apikey={FMP_API_KEY}"
  response = requests.get(url)
  data = response.json()
  try:
    results = data[0]
    financials = {
      "date": results["date"],
      "revenue": results["revenue"],
      "gross profit": results["grossProfit"],
      "net Income": results["netIncome"],
      "ebitda": results["ebitda"],
      "EPS": results["eps"],
      "EPS diluted":results["epsdiluted"]
    }
    return data, financials
  except (IndexError, KeyError):
    return {"error": f"Could not fetch financials for symbol: {symbol}"}

## Data provided by the Income Statement endpoint follows the following fomrat:
# [
# 	{
# 		"date": "2022-09-24",
# 		"symbol": "AAPL",
# 		"reportedCurrency": "USD",
# 		"cik": "0000320193",
# 		"fillingDate": "2022-10-28",
# 		"acceptedDate": "2022-10-27 18:01:14",
# 		"calendarYear": "2022",
# 		"period": "FY",
# 		"revenue": 394328000000,
# 		"costOfRevenue": 223546000000,
# 		"grossProfit": 170782000000,
# 		"grossProfitRatio": 0.4330963056,
# 		"researchAndDevelopmentExpenses": 26251000000,
# 		"generalAndAdministrativeExpenses": 0,
# 		"sellingAndMarketingExpenses": 0,
# 		"sellingGeneralAndAdministrativeExpenses": 25094000000,
# 		"otherExpenses": -334000000,
# 		"operatingExpenses": 51345000000,
# 		"costAndExpenses": 274891000000,
# 		"interestIncome": 2825000000,
# 		"interestExpense": 2931000000,
# 		"depreciationAndAmortization": 11104000000,
# 		"ebitda": 130541000000,
# 		"ebitdaratio": 0.3310467428,
# 		"operatingIncome": 119437000000,
# 		"operatingIncomeRatio": 0.302887444,
# 		"totalOtherIncomeExpensesNet": -334000000,
# 		"incomeBeforeTax": 119103000000,
# 		"incomeBeforeTaxRatio": 0.3020404333,
# 		"incomeTaxExpense": 19300000000,
# 		"netIncome": 99803000000,
# 		"netIncomeRatio": 0.2530964071,
# 		"eps": 6.15,
# 		"epsdiluted": 6.11,
# 		"weightedAverageShsOut": 16215963000,
# 		"weightedAverageShsOutDil": 16325819000,
# 		"link": "https://www.sec.gov/Archives/edgar/data/320193/000032019322000108/0000320193-22-000108-index.htm",
# 		"finalLink": "https://www.sec.gov/Archives/edgar/data/320193/000032019322000108/aapl-20220924.htm"
# 	}
# ]

In [25]:
get_stock_price("TSLA")

{'error': 'Invalid symbol or data not available for TSLA'}

In [26]:
get_company_financials("MPC")

{'error': 'Invalid symbol or data not available for MPC'}

In [27]:
get_income_statement("NVDA")

{'error': 'Could not fetch financials for symbol: NVDA'}

In [33]:
get_key_metrics_ttm("NVDA")

{'error': 'Invalid symbol or data not available for NVDA'}

In [34]:
stock_price_tool = FunctionTool.from_defaults(fn=get_stock_price)
company_financials_tool = FunctionTool.from_defaults(fn=get_company_financials)
key_metrics_ttm_tool = FunctionTool.from_defaults(fn=get_key_metrics_ttm)
tool_income_statement = FunctionTool.from_defaults(fn=get_income_statement)

In [36]:
from llama_index.core.agent import FunctionCallingAgent

claude_agent = FunctionCallingAgent.from_tools(
    [stock_price_tool, key_metrics_ttm_tool, tool_income_statement],
    llm=anthropic_llm,
    verbose=True,
    allow_parallel_tool_calls=True,
)

ImportError: cannot import name 'FunctionCallingAgent' from 'llama_index.core.agent' (/usr/local/lib/python3.12/dist-packages/llama_index/core/agent/__init__.py)

In [ ]:
from llama_index.core.agent import FunctionCallingAgent

oai_agent = FunctionCallingAgent.from_tools(
    [stock_price_tool, company_financials_tool, key_metrics_ttm_tool, tool_income_statement],
    llm=openai_llm,
    verbose=True,
    allow_parallel_tool_calls=True,
)

In [ ]:
query= "Give me the current price of MPC"
response = claude_agent.chat(query)
print(str(response))